# Dashboard: Portfolio Highlights

Reads each industry's business-metric table and renders one polished
Plotly chart per vertical — a quick "here's what this demo environment
covers" walkthrough across life sciences, healthcare, pharma,
agriculture, financial services, and digital marketing. Requires the
corresponding generator + transformation notebooks to have run first —
see `notebooks/README.md` for the full list.

Requires `plotly` and `pandas` (see `requirements.txt` in this folder).

In [ ]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import plotly.express as px

spark = SparkSession.builder.getOrCreate()
TEMPLATE = "plotly_white"

## Life Sciences — Trial Pipeline Health by Phase

In [ ]:
trials_pd = spark.table("clinical_trials.trial_pipeline_health_summary_v1").orderBy(F.desc("trial_count")).toPandas()

fig = px.bar(
    trials_pd, x="phase", y="trial_count", color="termination_rate_pct",
    color_continuous_scale="Blues",
    title="Life Sciences — Trial Count and Termination Rate by Phase",
)
fig.update_layout(template=TEMPLATE, height=400)
fig.show()

## Healthcare — Readmission Rate by Diagnosis

In [ ]:
readmit_pd = (
    spark.table("medicare_claims.readmission_risk_summary_v1")
    .orderBy(F.desc("total_admissions"))
    .limit(15)
    .toPandas()
    .sort_values("total_admissions")
)

fig = px.bar(
    readmit_pd, x="total_admissions", y="primary_diagnosis_code", orientation="h",
    color="readmission_rate_pct", color_continuous_scale="Reds",
    title="Healthcare — Top 15 Diagnoses by Admission Volume (colored by 30-day Readmission Rate)",
)
fig.update_layout(template=TEMPLATE, height=500, yaxis_title="ICD-9 diagnosis code")
fig.show()

## Pharma — Top Drug/Reaction Safety Signals

In [ ]:
signals_pd = (
    spark.table("pharmacovigilance.drug_safety_signal_summary_v1")
    .orderBy(F.desc("report_count"))
    .limit(15)
    .toPandas()
)
signals_pd["drug_reaction"] = signals_pd["drug_name"] + " → " + signals_pd["reaction_term"]

fig = px.bar(
    signals_pd.sort_values("report_count"), x="report_count", y="drug_reaction", orientation="h",
    color="serious_rate_pct", color_continuous_scale="OrRd",
    title="Pharma — Top 15 Drug/Reaction Safety Signals",
)
fig.update_layout(template=TEMPLATE, height=500, yaxis_title="")
fig.show()

## Agriculture — Biggest Crop Yield Movers

In [ ]:
yield_pd = (
    spark.table("agriculture.crop_yield_trend_summary_v1")
    .filter("yield_change_pct IS NOT NULL")
    .withColumn("state_commodity", F.concat_ws(" — ", "state", "commodity"))
    .orderBy(F.desc(F.abs("yield_change_pct")))
    .limit(15)
    .toPandas()
    .sort_values("yield_change_pct")
)

fig = px.bar(
    yield_pd, x="yield_change_pct", y="state_commodity", orientation="h",
    color="yield_change_pct", color_continuous_scale="RdYlGn", color_continuous_midpoint=0,
    title="Agriculture — Biggest Yield Movers, 2012 to 2017 Census",
)
fig.update_layout(template=TEMPLATE, height=500, yaxis_title="", coloraxis_showscale=False)
fig.show()

## Financial Services — Fraud Rate by Amount Bucket

In [ ]:
fraud_pd = (
    spark.table("payments.fraud_risk_summary_v1")
    .groupBy("amount_bucket")
    .agg(
        F.sum("transaction_count").alias("transaction_count"),
        F.sum("fraud_count").alias("fraud_count"),
    )
    .withColumn("fraud_rate_pct", F.round(F.col("fraud_count") * 100.0 / F.col("transaction_count"), 3))
    .toPandas()
)
bucket_order = ["0-10", "10-50", "50-100", "100-500", "500+"]
fraud_pd["amount_bucket"] = pd.Categorical(fraud_pd["amount_bucket"], categories=bucket_order, ordered=True)
fraud_pd = fraud_pd.sort_values("amount_bucket")

fig = px.bar(
    fraud_pd, x="amount_bucket", y="fraud_rate_pct",
    title="Financial Services — Fraud Rate by Transaction Amount Bucket",
)
fig.update_layout(template=TEMPLATE, height=400)
fig.show()

## Digital Marketing — Customer Segment Distribution

In [ ]:
segments_pd = (
    spark.table("retail.customer_rfm_segments_v1")
    .groupBy("segment")
    .agg(F.count("*").alias("customers"), F.round(F.avg("monetary"), 2).alias("avg_monetary"))
    .toPandas()
)

fig = px.pie(
    segments_pd, names="segment", values="customers",
    title="Digital Marketing — Customer Segment Distribution (RFM)",
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.update_layout(template=TEMPLATE, height=450)
fig.show()